# 09 — Genre ablation: clean categories vs noisy shelves (TF-IDF)

**The point.** The UCSD `goodreads_book_genres_initial` "genres" are themselves
keyword-extracted from popular shelves into ~8 broad classes — so **genre ⊂ shelves**, and we
already embed `shelves`. So the interesting question is *not* "does genre **add** signal" (it
barely can — it's derived from shelves) but:

> **Is a _clean_ genre label _better than_ the _raw, noisy_ shelves?**

Raw `popular_shelves` carry organizational junk (`to-read`, `owned`, `favorites`,
`currently-reading`) that dilutes the genre signal. `genre` is the denoised version. This is a
**denoising** test, run on **TF-IDF** so it needs **no re-embedding**. Only worth re-embedding
bge later if a real delta shows up here.

**Configs (document fields):**
- `title+plot` — floor, no categories
- `+shelves` — **baseline** (raw noisy categories)
- `+genre` — genre **replaces** shelves (clean categories)
- `+shelves+genre` — both (is genre redundant on top of shelves?)

Prereq: download `goodreads_book_genres_initial.json.gz` → `data/` (the cell below
auto-builds the `genre` column from it in-memory; it does **not** touch `catalog.parquet`).

In [1]:
import glob, os
import numpy as np, pandas as pd
from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.genres import stream_genres
from book_recsys.features.document import build_documents
from book_recsys.features.vectorize import tfidf_matrix
from book_recsys.models.content.content import ContentRecommender
from book_recsys.eval.harness import build_user_histories, build_relevance, evaluate_per_user
from book_recsys.eval.bootstrap import bootstrap_ci, paired_bootstrap

def find(name):
    for p in (name, f'artifacts/{name}', f'../artifacts/{name}'):
        if glob.glob(p): return glob.glob(p)[0]
    raise FileNotFoundError(name)
REPORTS = next((p for p in ('reports', '../reports') if os.path.isdir(p)), 'reports')
os.makedirs(REPORTS, exist_ok=True)
print('reports ->', os.path.abspath(REPORTS))

reports -> /Users/mayadeneva/Documents/uni/book_recsys/reports


In [2]:
sample  = pd.read_parquet(find('sample.parquet'))
catalog = pd.read_parquet(find('catalog.parquet'))
book_ids = catalog['book_id'].tolist()
print('sample', sample.shape, '| catalog', catalog.shape)

sample (12618544, 4) | catalog (468628, 9)


### Attach the `genre` column (in-memory, from the downloaded file)

In [4]:
# Build genre in-memory so we never clobber catalog.parquet. If you've already run
# scripts/enrich_catalog_genres.py, the column is reused as-is.
if 'genre' not in catalog.columns:
    raw = next(iter(glob.glob('data/goodreads_book_genres_initial.json*')
                    + glob.glob('../data/goodreads_book_genres_initial.json*')), None)
    if raw is None:
        raise FileNotFoundError(
            'Download goodreads_book_genres_initial.json.gz to data/ first '
            '(https://mengtingwan.github.io/data/goodreads.html).')
    catalog['genre'] = catalog['book_id'].map(stream_genres(raw)).fillna('')
have = int((catalog['genre'].str.len() > 0).sum())
print(f'genre present for {have:,}/{len(catalog):,} books')
print('sample genres:', catalog.loc[catalog['genre'].str.len() > 0, 'genre'].head(3).tolist())

genre present for 468,628/468,628 books
sample genres: ['fantasy, paranormal, fiction, mystery, thriller, crime', 'history, historical fiction, biography, mystery, thriller, crime, fiction', 'mystery, thriller, crime, fantasy, paranormal, history, historical fiction, biography']


### Split + eval users

In [5]:
# Full-catalog ranking over 468k books is the slow part. This ablation is a *paired*
# comparison (same users, same split, only the document fields change), so per-user variance
# cancels and modest N already resolves the differences. 1500 ~= 15-25 min on a Mac (CPU);
# raise for tighter CIs. For the final report row, re-run at higher N.
N_USERS = 1500
train, holdout = leave_last_n_out(sample, n=1)
rng = np.random.default_rng(0)
test_users = rng.choice(holdout['user_id'].unique(), size=N_USERS, replace=False)
histories = build_user_histories(train[train['user_id'].isin(test_users)])
relevance = build_relevance(holdout[holdout['user_id'].isin(test_users)])
print(len(relevance), 'test users')

1500 test users


### Run the four TF-IDF configs

In [6]:
configs = {
    'title+plot':       ('title', 'plot'),
    '+shelves (base)':  ('title', 'plot', 'shelves'),
    '+genre (replaces)':('title', 'plot', 'genre'),
    '+shelves+genre':   ('title', 'plot', 'shelves', 'genre'),
}
per_user, results = {}, {}
for name, fields in configs.items():
    matrix, _ = tfidf_matrix(build_documents(catalog, fields=fields))
    rec = ContentRecommender(book_ids, matrix).fit()
    per_user[name] = evaluate_per_user(rec, histories, relevance, k=10)
    results[name] = {m: float(np.mean(v)) for m, v in per_user[name].items()}
    print(f"{name:20s}", {k: round(v, 4) for k, v in results[name].items()})
tbl = pd.DataFrame(results).T[['recall@10', 'ndcg@10', 'mrr']].round(4)
tbl

title+plot           {'recall@10': 0.004, 'ndcg@10': 0.0027, 'mrr': 0.0022}
+shelves (base)      {'recall@10': 0.0047, 'ndcg@10': 0.0027, 'mrr': 0.0021}
+genre (replaces)    {'recall@10': 0.004, 'ndcg@10': 0.0025, 'mrr': 0.002}
+shelves+genre       {'recall@10': 0.0053, 'ndcg@10': 0.0028, 'mrr': 0.002}


,recall@10,ndcg@10,mrr
title+plot,0.0040,0.0027,0.0022
+shelves (base),0.0047,0.0027,0.0021
+genre (replaces),0.0040,0.0025,0.0020
+shelves+genre,0.0053,0.0028,0.0020


### Significance — is `+genre` really different from `+shelves`?

NDCG@10 with a 95% bootstrap CI, plus a **paired bootstrap vs the `+shelves` baseline**. This
is the whole experiment: *sig. better* = clean genre beats noisy shelves; *tie (n.s.)* = a
clean 8-class label is as good as the full shelf text (a tidy finding on its own); *sig.
worse* = the shelf noise was actually carrying useful signal.

In [7]:
base = '+shelves (base)'
base_mean = float(np.mean(per_user[base]['ndcg@10']))
rows = []
for name in configs:
    s = per_user[name]['ndcg@10']
    m, lo, hi = bootstrap_ci(s)
    if name == base:
        vs = '— (baseline)'
    elif paired_bootstrap(per_user[base]['ndcg@10'], s)['significant']:
        vs = 'sig. better' if m > base_mean else 'sig. worse'
    else:
        vs = 'tie (n.s.)'
    rows.append({'config': name, 'ndcg@10': round(m, 4),
                 '95% CI': f'[{lo:.4f}, {hi:.4f}]', f'vs {base}': vs})
cidf = pd.DataFrame(rows).sort_values('ndcg@10', ascending=False)
cidf

,config,ndcg@10,95% CI,vs +shelves (base)
3,+shelves+genre,0.0028,"[0.0009, 0.0049]",tie (n.s.)
0,title+plot,0.0027,"[0.0007, 0.0051]",tie (n.s.)
1,+shelves (base),0.0027,"[0.0008, 0.0050]",— (baseline)
2,+genre (replaces),0.0025,"[0.0006, 0.0049]",tie (n.s.)


### Save

In [8]:
tbl.to_csv(f'{REPORTS}/study_genre_ablation.csv')
cidf.to_csv(f'{REPORTS}/study_genre_ablation_ci.csv', index=False)
print('saved -> study_genre_ablation.csv + study_genre_ablation_ci.csv')
# Read-off: paste ndcg@10 + the 'vs +shelves' verdict into reports/model_report.md section 4.
# Only re-embed bge with +genre if clean genre is *sig. better* than shelves here.

saved -> study_genre_ablation.csv + study_genre_ablation_ci.csv
